# Confirmation of the pre-registered inflation guard (dead zone 1.5) on the NUDT valid split (seed 1212)
Rule, prior, dead zone and adoption criteria are in `results/preregistered_guard.json`, written before this draw was dumped. Nothing is changed here.

In [1]:
import os, sys, json, hashlib
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 220)
from train.checks import replay
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.decision_signals import build_signal_table
T7 = ROOT / "tests/Test_Iteration_7/results"
PRE_PATH = T7 / "preregistered_guard.json"; PRE = json.loads(PRE_PATH.read_text())
print("pre-registration written", PRE["written"], "sha256", hashlib.sha256(PRE_PATH.read_bytes()).hexdigest()[:16])
PRIOR, DEAD, WEIGHT = 0.157, PRE["dead_zone"], 30
NAME, DUMP = "nudt_valid1212", "nudt_valid1212"
out = T7 / f"signals_ours_{NAME}.csv.gz"
if not out.exists():
    build_signal_table(T7 / "rich_dump" / DUMP, NAME, 4).to_csv(out, index=False)
raw = pd.read_csv(out)
photos = {NAME: replay.load_set(T7 / "rich_dump" / DUMP)}
levels = sr.far_levels(raw)
print(len(photos[NAME]), "photos; far level", round(levels.far_inl.mean(), 4), "prior", PRIOR)

pre-registration written 2026-11-15T12:01:02 sha256 59a48565f8a517e5


150 photos; far level 0.2453 prior 0.157


In [2]:
SCALES = [1.0, 1.25, 1.5, 2.0]
def running_factor(f):
    g = levels.assign(far_inl=levels.far_inl * f).sort_values("photo")
    k = np.arange(1, len(g) + 1)
    run = (WEIGHT * PRIOR + g.far_inl.cumsum().to_numpy()) / (WEIGHT + k)
    ratio = run / PRIOR
    return {(NAME, p): (r if r > DEAD else 1.0) for p, r in zip(g.photo, ratio)}

def ours(f, guard):
    factor = running_factor(f) if guard else {}
    rows = []
    for pid, photo in photos[NAME].items():
        s = factor.get((NAME, pid), 1.0)
        inflated = [np.round(np.asarray(q["inliers"], float) * f) for q in photo["queries"]]
        r = replay.decide_current(photo, inliers=inflated, bar=round(8 * s), escalation_bar=round(11 * s), agree_query_inliers=round(3 * s))
        rows.append(dict(data=NAME, photo=pid, stage=r["stage"], pts=int(r["points"]), err_sigma=r["err_sigma"], n_queries=r["n_queries"]))
    return pd.DataFrame(rows)

rows = []; KEEP = {}
for f in SCALES:
    for guard in [False, True]:
        d = ours(f, guard); KEEP[(f, guard)] = d
        s = cj.summary(d).loc[NAME]
        rows.append(dict(scale=f, guard=guard, score=s.score, wrong=int(s.wrong), answered=s.answered, queries=s.queries))
TAB = pd.DataFrame(rows)
display(pd.concat({"score": TAB.pivot_table(index="guard", columns="scale", values="score").round(2), "wrong": TAB.pivot_table(index="guard", columns="scale", values="wrong").astype(int)}, axis=1))

score                   wrong               
scale  1.00  1.25  1.50  2.00  1.00 1.25 1.50 2.00
guard                                             
False  51.6  51.6  48.8  45.6     3    4    7   11
True   51.6  52.4  52.0  52.0     3    2    2    2

In [3]:
gain = {f: cj.paired_gain(KEEP[(f, False)], KEEP[(f, True)], [NAME])[NAME] for f in SCALES}
wr = lambda f, g: int(TAB[(TAB.scale == f) & (TAB.guard == g)].wrong.iloc[0])
sc = lambda f, g: float(TAB[(TAB.scale == f) & (TAB.guard == g)].score.iloc[0])
verdict = dict(
    x1_ok=bool(wr(1.0, True) <= wr(1.0, False) and sc(1.0, True) - sc(1.0, False) >= -0.30),
    x15_ok=bool(wr(1.5, True) <= wr(1.5, False)),
    x2_ok=bool(wr(2.0, True) < wr(2.0, False)))
verdict["adopt"] = all(verdict.values())
TAB.to_csv(T7 / "ours_inflation_guard_valid1212.csv", index=False)
print(gain); print(verdict)

{1.0: '+0.00 [+0.00, +0.00]', 1.25: '+0.80 [-1.60, +3.60]', 1.5: '+3.20 [-0.01, +7.20]', 2.0: '+6.40 [+2.00, +11.60]'}
{'x1_ok': True, 'x15_ok': True, 'x2_ok': True, 'adopt': True}
